In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Dense, 
                                    Input, 
                                    LSTM, 
                                    Dropout, 
                                    Conv1D, 
                                    MaxPooling1D, 
                                    Flatten
                                )
from tensorflow.keras.optimizers import (Adam, 
                                         AdamW)
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import classification_report

In [2]:
data = pd.read_pickle("../data/processed/eeg_bandpass_ica.pkl")
data = data[data["shape"]==(64, 656)]

#### Step 1 : Setting Data in Appropriate Shape

In [3]:
X = np.stack(data["epoch_data_ICA"].values, axis = 0)
# Transpose to match (num_trials, n_samples, n_channels)
X = np.transpose(X, (0, 2, 1))

y = data["label"].values

print(f"Shape of X: {X.shape}")
print(f"Shape of y: {y.shape}")

Shape of X: (4083, 656, 64)
Shape of y: (4083,)


#### Step -2 : Normalize Data

In [4]:
# Global Normalization
X = (X-np.mean(X))/np.std(X)

In [5]:
X.shape[0]

4083

#### Step 3: One-hot Encode labels (for classification)

In [6]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
le.fit(y)

# Transform 
y_le = le.transform(y)

# Applying to_categorical 
y_le = to_categorical(y_le, num_classes= len(np.unique(y_le)))
print(f"Shape of y : {y_le.shape}")

Shape of y : (4083, 2)


#### Step 4. Creating tensorflow Dataset

In [7]:
data = tf.data.Dataset.from_tensor_slices((X, y_le))

#### Step 5: Splitting the Tensorflow Dataset into train, test and valid

In [8]:
train_split_ratio = 0.7
val_split_ratio = 0.15
test_split_ratio = 0.15

# Size of each split
train_size = int(train_split_ratio * X.shape[0])
val_size = int(val_split_ratio * X.shape[0])
test_size = int(test_split_ratio * X.shape[0])

# Shuffle the dataset first for a random split
data = data.shuffle(buffer_size= X.shape[0])

training_dataset = data.take(train_size)
val_dataset = data.take(val_size)
test_dataset = data.take(test_size)

print(f"Train dataset size: {len(list(training_dataset.as_numpy_iterator()))}")
print(f"Val dataset size: {len(list(val_dataset.as_numpy_iterator()))}")
print(f"Test dataset size: {len(list(test_dataset.as_numpy_iterator()))}")

Train dataset size: 2858
Val dataset size: 612
Test dataset size: 612


2025-10-02 20:42:45.128703: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2025-10-02 20:42:45.220288: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [9]:
training_dataset = training_dataset.batch(32)
val_dataset = val_dataset.batch(32)
test_dataset = test_dataset.batch(32)

In [10]:
for index, epoch in enumerate(training_dataset.take(1)):
    print(epoch[0].shape)

(32, 656, 64)


2025-10-02 20:42:45.342085: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


#### CNN MODEL ARCHITECTURE - USING MODEL SUBCLASSING

In [11]:
class CNNModel(tf.keras.Model):

    def __init__(self, num_classes):

        super(CNNModel, self).__init__()
        
        # Convolution layer 1
        self.conv_layer_1 = Conv1D(filters= 64, kernel_size= 3, strides= 2, activation= 'relu')
        self.max_pooling_layer_1 = MaxPooling1D(pool_size = 2)
        
        # Convolution layer 2
        self.conv_layer_2 = Conv1D(filters= 128, kernel_size= 3, strides= 1, activation= 'relu')
        self.max_pooling_layer_2 = MaxPooling1D(pool_size = 2)

        # flatten layer1
        self.flatten = Flatten()

        # Dense layer
        self.dense_1 = Dense(units = 256, activation= "leaky_relu") # Denselayer1
        self.dropout_1 = Dropout(0.2)
        self.dense_2 = Dense(units = 128, activation= "leaky_relu") # Denselayer2
        self.dropout_2 = Dropout(0.2)
        self.dense_3 = Dense(units = 64, activation= "relu") # Denselayer3

        # Output layer 
        self.out = Dense(units = num_classes, activation= "softmax")


    def call(self, inputs):
        x = inputs
        x = self.conv_layer_1(x) # conv_layer_1
        x = self.max_pooling_layer_1(x) # max_pooling_layer_1

        x = self.conv_layer_2(x) # conv_layer_2
        #x = self.max_pooling_layer_2(x) # max_pooling_layer_2

        x = self.flatten(x) # flatten layer

        x = self.dense_1(x) # Denselayer1
        x = self.dropout_1(x) # DropOut layer1
        x = self.dense_2(x) # Denselayer2
        x = self.dropout_2(x) # DropOut layer 2
        x = self.dense_3(x) # Denselayer3

        out = self.out(x) # final output layer

        return out
    
# CNN Architecture
basic_cnn_model = CNNModel(num_classes=2)

In [12]:
print(basic_cnn_model.summary())

# Compile the Model
basic_cnn_model.compile(optimizer= Adam(), 
                        loss = "categorical_crossentropy", 
                        metrics = ["f1_score"])

Model: "cnn_model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

None


In [26]:
history = basic_cnn_model.fit(
    training_dataset,
    epochs = 100, 
    batch_size = 32, 
    validation_data = val_dataset
)

Epoch 1/100
90/90 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - f1_score: 0.9549 - loss: 0.1247 - val_f1_score: 0.9771 - val_loss: 0.0755
Epoch 2/100
90/90 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - f1_score: 0.9584 - loss: 0.1025 - val_f1_score: 0.9574 - val_loss: 0.1093
Epoch 3/100
90/90 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - f1_score: 0.9608 - loss: 0.1024 - val_f1_score: 0.9869 - val_loss: 0.0711
Epoch 4/100
90/90 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - f1_score: 0.9591 - loss: 0.1090 - val_f1_score: 0.9737 - val_loss: 0.0689
Epoch 5/100
90/90 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - f1_score: 0.9727 - loss: 0.0677 - val_f1_score: 0.9869 - val_loss: 0.0329
Epoch 6/100
90/90 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - f1_score: 0.9825 - loss: 0.0438 - val_f1_score: 0.9853 - val_loss: 0.0453
Epoch 7/100
90/90 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - f1_score: 0.9696 - loss: 0.0789 - val_f1_score: 0.9869 - val_loss: 0.0328
Epoch 8/100
90/90 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - f1_score: 0.9829 - loss: 0.0530 - val_f1_score: 0.

In [27]:
# Test Predictions
y_val_pred = basic_cnn_model.predict(val_dataset)
y_val_pred = np.argmax(y_val_pred, axis = 1)

# Original Predictions
y_val_original = np.concatenate([y.numpy() for x, y in val_dataset])
y_val_original = np.argmax(y_val_original, axis = 1)

20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step


2025-10-02 20:50:26.353054: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [32]:
print(classification_report(y_val_original, y_val_pred))

              precision    recall  f1-score   support

           0       0.47      0.46      0.47       313
           1       0.45      0.46      0.46       299

    accuracy                           0.46       612
   macro avg       0.46      0.46      0.46       612
weighted avg       0.46      0.46      0.46       612



# Test Predictions

In [33]:
# Test Predictions
y_test_pred = basic_cnn_model.predict(test_dataset)
y_test_pred = np.argmax(y_test_pred, axis = 1)

# Original Predictions
y_test_original = np.concatenate([y.numpy() for x, y in test_dataset])
y_test_original = np.argmax(y_test_original, axis = 1)

20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step


In [34]:
print(classification_report(y_test_original, y_test_pred))

              precision    recall  f1-score   support

           0       0.52      0.53      0.52       313
           1       0.49      0.48      0.49       299

    accuracy                           0.51       612
   macro avg       0.51      0.51      0.51       612
weighted avg       0.51      0.51      0.51       612

